<a href="https://colab.research.google.com/github/deacs11/CrewAI_Contract_Clause_Risk_Assessment/blob/main/CrewAI_Contract_Clause_Risk_Assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 1. Install necessary libraries
# Installs crewai, tools, OpenAI client, Colab module, and PyMuPDF for PDF handling.
!pip install crewai crewai-tools langchain-openai google-colab pymupdf -q

print("Library installation completed! (PyMuPDF added)")

Library installation completed! (PyMuPDF added)


In [ ]:
# @title 2. Import modules and configure API Keys from Secrets
import os
from google.colab import userdata # To read Colab secrets
from crewai import Agent, Task, Crew, Process
# No external tools needed for this basic version (Serper, etc.)
# from crewai_tools import SerperDevTool, WebsiteSearchTool
from langchain_openai import ChatOpenAI
import re # Regular expressions might be useful for basic parsing

# --- API KEY CONFIGURATION FROM COLAB SECRETS ---
print("Attempting to load API Keys from Secrets...")
try:
    # Reads the OpenAI key from the secret 'OPENAI_API_KEY'
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    print("-> OpenAI API Key environment variable set successfully.")
except Exception as e:
    print(f"!!! Error loading OpenAI API Key from Secrets: {e}")
    print("!!! Make sure you have set 'OPENAI_API_KEY' correctly in the Colab Secrets panel")
    print("!!! and enabled 'Notebook access'.")

# Debug block to verify the key
print("-" * 20)
retrieved_key = os.environ.get("OPENAI_API_KEY")
if retrieved_key:
    print(f"OpenAI Key FOUND in environment: '{retrieved_key[:5]}...{retrieved_key[-4:]}'")
else:
    print("!!! OpenAI Key NOT FOUND in environment.")
    print("!!! Check Colab Secrets and re-run this cell.")
print("-" * 20)

Attempting to load API Keys from Secrets...
-> OpenAI API Key environment variable set successfully.
--------------------
OpenAI Key FOUND in environment: 'sk-pr...DKkA'
--------------------


In [ ]:
# @title 3. Upload contracts & define text (Handles PDF and TXT, multiple files)

import os
import fitz  # PyMuPDF library for PDF handling

# --- Instructions for User ---
print("Upload your contract files to Colab session storage, then paste their paths below.")
print("1. Click the 'Folder' icon in the left sidebar.")
print("2. Click 'Upload to session storage' (upward-arrow icon).")
print("3. Upload all contract files (PDF or TXT supported).")
print("4. Right-click each uploaded file and select 'Copy path'.")
print("5. Paste all paths into the field below, separated by commas, then run this cell.")
print("-" * 20)

# --- Colab Form Field: comma-separated file paths ---
uploaded_file_paths = "/content/contract_a.pdf, /content/contract_b.txt"  # @param {type:"string"}

def extract_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".pdf":
        print(f"  Detected PDF. Extracting text with PyMuPDF...")
        doc = fitz.open(file_path)
        page_count = len(doc)
        text = "\n".join(doc.load_page(i).get_text() for i in range(page_count))
        doc.close()
        print(f"  Extracted text from {page_count} page(s).")
        return text
    elif ext == ".txt":
        print(f"  Detected TXT. Reading as plain text...")
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    else:
        print(f"  WARNING: Unsupported extension '{ext}'. Attempting plain-text read...")
        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                return f.read()
        except Exception as e:
            print(f"  Failed to read as plain text: {e}")
            return ""

contracts = []
paths = [p.strip() for p in uploaded_file_paths.split(",") if p.strip()]

for path in paths:
    filename = os.path.basename(path)
    print(f"\nLoading: {path}")
    if not os.path.exists(path):
        print(f"  ERROR: File not found at '{path}'. Skipping.")
        continue
    try:
        text = extract_text(path)
        if not text:
            print(f"  WARNING: No text extracted from '{filename}'. Skipping.")
            continue
        if len(text) < 100:
            print(f"  WARNING: Extracted text is very short — check file quality (e.g. scanned image?).")
        print(f"  Preview: {repr(text[:200].strip())}...")
        contracts.append({"filename": filename, "text": text})
    except Exception as e:
        print(f"  ERROR processing '{filename}': {e}. Skipping.")

print("\n" + "*"*70)
print("!!! DISCLAIMER: Contract Analysis Assistant !!!")
print("This tool provides automated analysis to *assist* in identifying potential")
print("areas of interest or risk in contract text based on common patterns.")
print("It DOES NOT provide legal advice. The output is generated by an AI")
print("and may contain errors, omissions, or misinterpretations (especially with complex PDF layouts).")
print("All findings MUST be reviewed by a qualified legal professional")
print("before making any decisions or taking any action.")
print("Do not rely solely on this tool for legal assessment.")
print("*"*70)

if not contracts:
    print("\n!!! CRITICAL: No contracts were loaded. Check file paths and re-run this cell.")
else:
    print(f"\n{len(contracts)} contract(s) loaded and ready for analysis:")
    for c in contracts:
        print(f"  - {c['filename']}  ({len(c['text'])} chars)")


In [ ]:
# @title 4. Select LLM and initialize Tools

# --- CHOOSE THE LANGUAGE MODEL (LLM) ---
try:
    # GPT-4 is highly recommended for the nuance required in legal text analysis
    llm = ChatOpenAI(model="gpt-4-turbo")
    # llm = ChatOpenAI(model="gpt-3.5-turbo") # Might struggle significantly with legal nuance
    print(f"LLM ({llm.model_name}) initialized successfully.")
except Exception as e:
    print(f"!!! Error initializing ChatOpenAI: {e}")
    print("!!! Verify OpenAI API key.")
    raise

# --- INITIALIZE TOOLS ---
# For this version, we assume the contract text is passed directly.
# No external search or web reading tools are needed by the core agents.
available_tools = []
print("No external tools initialized for this crew.")

LLM (gpt-4-turbo) initialized successfully.
No external tools initialized for this crew.


In [ ]:
# @title 5. Define agents for Contract Analysis Crew

print("Defining Contract Analysis Agents...")

if 'llm' not in locals() or llm is None:
     raise ValueError("LLM not initialized.")

# --- Agent 1: Contract Parser ---
# Basic parsing attempt. A more robust solution might involve dedicated libraries or regex.
contract_parser = Agent(
    role='Legal Document Structure Analyst',
    goal='Segment the provided contract text into individually identifiable clauses or sections. Preserve numbering if present. Output a list where each item represents a distinct clause or paragraph.',
    backstory=(
        "You are meticulous at analyzing document structure. You identify distinct paragraphs or numbered items as separate clauses. "
        "Your goal is to break down the contract into its core components for easier analysis by other specialists. Handle potential variations in numbering or formatting."
    ),
    tools=[],
    llm=llm,
    allow_delegation=False,
    verbose=True,
    max_iter=3
)
print("- Agent 'contract_parser' defined.")

# --- Agent 2: Clause classifier (optional but helpful) ---
# This helps downstream agents focus, but adds a step. Can be removed if needed.
clause_classifier = Agent(
    role='Legal Clause Taxonomy Expert',
    goal='For each segmented clause provided, assign a likely category based on its content (e.g., Services, Term, Payment, Confidentiality, Warranty, Liability, Indemnification, Termination, Governing Law, Miscellaneous).',
    backstory=(
        "You have a deep understanding of common contract structures and the purpose of different clauses. "
        "Based on keywords and context, you can accurately categorize most standard contract provisions. "
        "Acknowledge if a clause seems miscellaneous or hard to classify."
    ),
    tools=[],
    llm=llm,
    allow_delegation=False,
    verbose=True,
    max_iter=5
)
print("- Agent 'clause_classifier' defined.")

# --- Agent 3: Risk Pattern Detector ---
risk_pattern_detector = Agent(
    role='Contract Risk Spotter',
    goal='Scan each provided contract clause text. Identify and flag clauses containing specific high-risk patterns or keywords commonly associated with unfavorable terms. Focus on patterns like: unlimited liability, broad indemnification obligations imposed ON THE CLIENT, weak warranties (e.g., extensive disclaimers), ambiguous or one-sided termination rights, automatic renewals without clear opt-out, non-standard governing law/jurisdiction choices, overly broad confidentiality obligations.',
    backstory=(
        "You are trained to recognize red flags in contract language based on common legal and business risk concerns. "
        "You meticulously compare clause text against a known set of potentially problematic phrases and concepts. "
        "You don't interpret the overall fairness, just flag the presence of specific predefined risk patterns."
    ),
    tools=[],
    llm=llm, # This agent heavily relies on the LLM's pattern matching based on the goal description
    allow_delegation=False,
    verbose=True,
    max_iter=7
)
print("- Agent 'risk_pattern_detector' defined.")

# --- Agent 4: Ambiguity Identifier ---
ambiguity_identifier = Agent(
    role='Clarity and Precision Analyst',
    goal='Review each contract clause for ambiguous language, undefined critical terms (e.g., "reasonable efforts", "material breach" without definition), potentially contradictory statements, or overly broad phrasing that could lead to future disputes or misinterpretations. Flag clauses requiring clarification.',
    backstory=(
        "You focus intensely on linguistic precision and clarity in legal documents. You hunt for words or phrases that lack specific definition "
        "where one might be needed, or sentences structured in a way that allows for multiple interpretations. Your aim is to flag areas needing refinement for certainty."
    ),
    tools=[],
    llm=llm,
    allow_delegation=False,
    verbose=True,
    max_iter=5
)
print("- Agent 'ambiguity_identifier' defined.")

# --- Agent 5: Review Brief Generator ---
review_brief_generator = Agent(
    role='Legal Review Summarizer',
    goal='Consolidate all flagged clauses identified by the Risk Pattern Detector and the Ambiguity Identifier into a single, structured "Review Brief". For each flagged item, clearly state the clause number (or reference), the clause text, and the specific reason(s) it was flagged (e.g., "Risk Pattern: Unlimited Liability", "Ambiguity: Undefined term \'material\'"). Organize the brief for efficient review by a legal professional.',
    backstory=(
        "You excel at synthesizing analytical findings into clear, actionable summaries for busy professionals. "
        "You organize flagged items logically, provide necessary context (clause text and reason), and ensure the output is easy to scan and understand. "
        "The goal is to facilitate a focused human review."
    ),
    tools=[],
    llm=llm,
    allow_delegation=False,
    verbose=True,
    max_iter=3
)
print("- Agent 'review_brief_generator' defined.")

print("All contract analysis agents defined.")

Defining Contract Analysis Agents...
- Agent 'contract_parser' defined.
- Agent 'clause_classifier' defined.
- Agent 'risk_pattern_detector' defined.
- Agent 'ambiguity_identifier' defined.
- Agent 'review_brief_generator' defined.
All contract analysis agents defined.


In [ ]:
# @title 6. Define task-creation function for Contract Analysis Crew

def create_tasks(contract_text):
    """Build and return a fresh list of 5 analysis tasks for the given contract text."""
    for name in ['contract_parser', 'clause_classifier', 'risk_pattern_detector',
                 'ambiguity_identifier', 'review_brief_generator']:
        if name not in globals():
            raise ValueError(f"Agent '{name}' not defined. Run Cell 5 first.")

    # --- Task 1: Parse Contract into Clauses ---
    task_parse_contract = Task(
        description=(
            f"Process the following contract text provided in the initial input:\n---\n{contract_text}\n---\n"
            f"Segment this text into a list of distinct clauses or sections. Attempt to preserve any existing numbering (e.g., '1.', 'Section 2.1')."
            f"Output should be a structured representation (like a numbered list or list of strings) of these segmented clauses."
        ),
        agent=contract_parser,
        expected_output=(
            "A list containing the text of each identified clause or section from the input contract. "
            "Example: ['1. SERVICES. Provider agrees...', '2. TERM. This Agreement shall...', ...]"
        )
    )

    # --- Task 2: Classify Clauses ---
    task_classify_clauses = Task(
        description=(
            "Take the list of segmented contract clauses (provided from the previous task's context). "
            "For each clause, determine its likely primary category (e.g., Services, Term, Payment, Confidentiality, Warranty, Limitation of Liability, Indemnification, Termination, Governing Law, Entire Agreement/Miscellaneous). "
            "Present the output as a list where each item contains the original clause text and its assigned category."
        ),
        agent=clause_classifier,
        expected_output=(
            "A list, where each element corresponds to a clause and includes the clause text and its assigned category. "
            "Example: [{\'clause_text\': \'1. SERVICES...\', \'category\': \'Services\'}, {\'clause_text\': \'6. LIMITATION OF LIABILITY...\', \'category\': \'Limitation of Liability\'}, ...]"
        ),
        context=[task_parse_contract]
    )

    # --- Task 3: Detect Risk Patterns ---
    task_detect_risks = Task(
        description=(
            "Review the list of classified contract clauses (provided in context). "
            "Scan the text of *each* clause specifically for the presence of predefined high-risk patterns or keywords (as defined in the Risk Spotter agent's goal, e.g., unlimited liability, broad client indemnification, weak warranties, ambiguous termination, auto-renewal issues, non-standard jurisdiction). "
            "Output a list containing ONLY the clauses that were flagged, including the clause text and the specific risk pattern(s) identified for each flagged clause."
        ),
        agent=risk_pattern_detector,
        expected_output=(
            "A list containing only the clauses identified as potentially risky. Each item in the list should include the clause text and the specific reason(s)/pattern(s) why it was flagged. If no risks are found, the list should be empty. "
            "Example: [{\'clause_text\': \'6. LIMITATION OF LIABILITY...\', \'risk_flag\': \'Broad disclaimer of implied warranties\'}, {\'clause_text\': \'7. INDEMNIFICATION...\', \'risk_flag\': \'Client indemnifies Provider broadly\'}]"
        ),
        context=[task_classify_clauses]
    )

    # --- Task 4: Identify Ambiguities ---
    task_identify_ambiguities = Task(
        description=(
            "Review the list of classified contract clauses (provided in context). "
            "Analyze the text of *each* clause for potential ambiguities, undefined critical terms, vague language, or contradictions that might require clarification. "
            "Output a list containing ONLY the clauses that were flagged for ambiguity, including the clause text and a brief explanation of the ambiguity identified in each flagged clause."
        ),
        agent=ambiguity_identifier,
        expected_output=(
            "A list containing only the clauses identified as potentially ambiguous. Each item should include the clause text and a brief note explaining the source of ambiguity. If no ambiguities are found, the list should be empty. "
            "Example: [{\'clause_text\': \'1. SERVICES... Provider shall determine the method...\', \'ambiguity_flag\': \'Term \\\"method, details, and means\\\" is vague\'}, {\'clause_text\': \'8. TERMINATION... materially breaches...\', \'ambiguity_flag\': \'Term \\\"materially breaches\\\" is undefined\'}]"
        ),
        context=[task_classify_clauses]
    )

    # --- Task 5: Generate Review Brief ---
    task_generate_brief = Task(
        description=(
            "Consolidate the findings from the risk detection task and the ambiguity identification task (outputs provided in context) into a structured 'Contract Review Brief'. "
            "The brief should have two main sections: 'Potential Risk Flags' and 'Potential Ambiguities/Clarifications Needed'. "
            "Under each section, list the items identified previously, including the original clause text (or a clear reference like clause number if consistently available from parsing) and the specific reason it was flagged (risk pattern or ambiguity explanation). "
            "Format the output clearly using Markdown for easy reading by a legal professional. Include the disclaimer about this being an assistant tool."
        ),
        agent=review_brief_generator,
        expected_output=(
            "A well-formatted Markdown document titled 'Contract Review Brief (AI Assisted)'. "
            "It must include the disclaimer. "
            "It should contain two distinct sections: 'Potential Risk Flags' and 'Potential Ambiguities/Clarifications Needed'. "
            "Each section should list the relevant flagged clauses with their text and the reason for flagging. "
            "The output should be ready for human review."
        ),
        context=[task_detect_risks, task_identify_ambiguities]
    )

    return [task_parse_contract, task_classify_clauses, task_detect_risks,
            task_identify_ambiguities, task_generate_brief]

print("Task creation function 'create_tasks' defined.")


In [ ]:
# @title 7. Create and Run Contract Analysis Crew (Multi-Contract)

if 'contracts' not in dir() or not contracts:
    raise ValueError("No contracts loaded. Run Cell 3 first.")

for name in ['contract_parser', 'clause_classifier', 'risk_pattern_detector',
             'ambiguity_identifier', 'review_brief_generator']:
    if name not in dir():
        raise ValueError(f"Agent '{name}' not defined. Run Cell 5 first.")

if 'create_tasks' not in dir():
    raise ValueError("'create_tasks' not defined. Run Cell 6 first.")

results = {}

for i, contract in enumerate(contracts, 1):
    print(f"\n{'='*60}")
    print(f"  Analyzing contract {i}/{len(contracts)}: {contract['filename']}")
    print(f"{'='*60}")

    tasks = create_tasks(contract["text"])

    crew = Crew(
        agents=[contract_parser, clause_classifier, risk_pattern_detector,
                ambiguity_identifier, review_brief_generator],
        tasks=tasks,
        process=Process.sequential,
        memory=True,
        cache=True,
        verbose=True,
    )

    result = crew.kickoff()
    results[contract["filename"]] = result
    print(f"\n  Finished: {contract['filename']}")

print("\n\n*****************************************")
print(f"   ALL {len(contracts)} CONTRACT(S) ANALYZED!")
print("*****************************************")


In [ ]:
# @title 8. Display final review briefs

if 'results' not in dir() or not results:
    print("!!! No results found. Ensure Cell 7 executed correctly.")
else:
    for filename, result in results.items():
        print("\n" + "="*70)
        print(f"## AI-Assisted Contract Review Brief: {filename}")
        print("## REMINDER: FOR HUMAN REVIEW ONLY - NOT LEGAL ADVICE")
        print("="*70 + "\n")
        print(result)
        print("\n" + "*"*70)
        print("!!! IMPORTANT: The above output is AI-generated assistance.")
        print("!!! It MUST be reviewed thoroughly by a qualified legal professional.")
        print("!!! Do not rely on this output for legal decisions.")
        print("*"*70)
